# Projeto Airbnb Rio - Ferramenta de Previsão de Preço de Imóvel para pessoas comuns 

### Objetivo

Construir um modelo de previsão de preço que permita uma pessoa comum que possui um imóvel possa saber quanto deve cobrar pela diária do seu imóvel.

Ou ainda, para o locador comum, dado o imóvel que ele está buscando, ajudar a saber se aquele imóvel está com preço atrativo (abaixo da média para imóveis com as mesmas características) ou não.

### O que temos disponível, inspirações e créditos

As bases de dados foram retiradas do site kaggle: https://www.kaggle.com/allanbruno/airbnb-rio-de-janeiro

### Expectativas Iniciais

- Acredito que a sazonalidade pode ser um fator importante, visto que meses como dezembro costumam ser bem caros no RJ
- A localização do imóvel deve fazer muita diferença no preço, já que no Rio de Janeiro a localização pode mudar completamente as características do lugar (segurança, beleza natural, pontos turísticos)
- Adicionais/Comodidades podem ter um impacto significativo, visto que temos muitos prédios e casas antigos no Rio de Janeiro.

### Importar Bibliotecas

In [39]:
import pandas as pd
import pathlib

### Consolidar Base de Dados

In [40]:
# percorrer todos os arquivos da pasta
meses = {'jan': 1, 'fev':2, 'mar':3, 'abr':4, 'mai':5, 'jun':6, 'jul':7, 'ago':8, 'set':9, 'out':10, 'nov':11, 'dez':12}
caminho_base = pathlib.Path('dataset')

lista = []
base = pd.DataFrame()

for arquivo in caminho_base.iterdir():
    nome_mes = arquivo.name[:3]
    mes = meses[nome_mes]
    ano = arquivo.name[-8:]
    ano = int(ano.replace(".csv", ""))

    df = pd.read_csv(caminho_base/arquivo.name)
    df['ano'] = ano
    df['mes'] = mes
    lista.append(df)

base = pd.concat(lista, ignore_index=True)        # base de dados com todos os arquivos juntos
# display(base)
base.info(verbose=True, show_counts=True)


C:\Users\Andre\AppData\Local\Temp\ipykernel_964\2236680681.py:14: DtypeWarning: Columns (0: monthly_price, 1: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_base/arquivo.name)
C:\Users\Andre\AppData\Local\Temp\ipykernel_964\2236680681.py:14: DtypeWarning: Columns (0: weekly_price, 1: monthly_price, 2: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_base/arquivo.name)
C:\Users\Andre\AppData\Local\Temp\ipykernel_964\2236680681.py:14: DtypeWarning: Columns (0: weekly_price, 1: monthly_price, 2: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_base/arquivo.name)
C:\Users\Andre\AppData\Local\Temp\ipykernel_964\2236680681.py:14: DtypeWarning: Columns (0: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_base/arquivo.name)
C:\Users\Andre\AppData\Local

<class 'pandas.DataFrame'>
RangeIndex: 902210 entries, 0 to 902209
Data columns (total 108 columns):
 #    Column                                        Non-Null Count   Dtype  
---   ------                                        --------------   -----  
 0    id                                            902210 non-null  int64  
 1    listing_url                                   902210 non-null  str    
 2    scrape_id                                     902210 non-null  int64  
 3    last_scraped                                  902210 non-null  str    
 4    name                                          900538 non-null  str    
 5    summary                                       860406 non-null  str    
 6    space                                         543496 non-null  str    
 7    description                                   884040 non-null  str    
 8    experiences_offered                           902210 non-null  str    
 9    neighborhood_overview                         

### Remover todas as colunas que contenham links (url)

In [41]:
# Identifica colunas que contêm 'url'
remover = [coluna for coluna in base.columns if 'url' in coluna.lower()]

# Deleta as colunas encontradas
base.drop(columns=remover, inplace=True)

print(f"Colunas removidas: {remover}")

Colunas removidas: ['listing_url', 'thumbnail_url', 'medium_url', 'picture_url', 'xl_picture_url', 'host_url', 'host_thumbnail_url', 'host_picture_url']


### Remover todas as colunas que contenham ID

In [42]:
remover = [coluna for coluna in base.columns if 'id' in coluna.lower()]

# Deleta as colunas encontradas
base.drop(columns=remover, inplace=True)

print(f"Colunas removidas: {remover}")

base.info(verbose=True, show_counts=True)

Colunas removidas: ['id', 'scrape_id', 'host_id', 'host_identity_verified']
<class 'pandas.DataFrame'>
RangeIndex: 902210 entries, 0 to 902209
Data columns (total 96 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   last_scraped                                  902210 non-null  str    
 1   name                                          900538 non-null  str    
 2   summary                                       860406 non-null  str    
 3   space                                         543496 non-null  str    
 4   description                                   884040 non-null  str    
 5   experiences_offered                           902210 non-null  str    
 6   neighborhood_overview                         481672 non-null  str    
 7   notes                                         284885 non-null  str    
 8   transit                                       476428 non-nu

### Remover colunas com muitas NAN

In [43]:
# Verifica a porcentagem de nulos por coluna
nulos = base.isnull().mean() * 100
colunas_muitos_nulos = nulos[nulos > 50].index.tolist()


# 2. Exclui do DataFrame
base = base.drop(columns=colunas_muitos_nulos)

# 3. Verifica o novo tamanho do DataFrame
print(f"Foram removidas {len(colunas_muitos_nulos)} colunas.")
print(f"Novo formato do DataFrame: {base.shape}")

Foram removidas 11 colunas.
Novo formato do DataFrame: (902210, 85)


### Remover colunas com valor único (todos iguais)

In [44]:
# Identifica colunas que têm apenas 1 valor único
colunas_inuteis = [col for col in base.columns if df[col].nunique() <= 1]

# 2. Exclui do DataFrame
base = base.drop(columns=colunas_inuteis)

# 3. Verifica o novo tamanho do DataFrame
print(f"Foram removidas {len(colunas_inuteis)} colunas.")
print(f"Novo formato do DataFrame: {base.shape}")

Foram removidas 6 colunas.
Novo formato do DataFrame: (902210, 79)


In [45]:
base.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 902210 entries, 0 to 902209
Data columns (total 79 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   last_scraped                                  902210 non-null  str    
 1   name                                          900538 non-null  str    
 2   summary                                       860406 non-null  str    
 3   space                                         543496 non-null  str    
 4   description                                   884040 non-null  str    
 5   neighborhood_overview                         481672 non-null  str    
 6   transit                                       476428 non-null  str    
 7   house_rules                                   457813 non-null  str    
 8   host_name                                     901750 non-null  str    
 9   host_since                                    901750 non-nu

### Ver colunas no excel

In [46]:
base.head(500).to_csv('verificar_colunas.csv', sep=';')

### Remover uma lista de colunas selecionadas manualmente

In [47]:
colunas_para_remover = ['last_scraped', 'name', 'summary','host_name', 'host_since', 'host_location', 'host_response_time', 'host_response_rate', 'host_total_listings_count', 'host_verifications','city', 'state', 'zipcode', 'country']
base = base.drop(columns=colunas_para_remover)

# 3. Verifica o novo tamanho do DataFrame
print(f"Foram removidas {len(colunas_para_remover)} colunas.")
print(f"Novo formato do DataFrame: {base.shape}")

Foram removidas 14 colunas.
Novo formato do DataFrame: (902210, 65)


### Tratar Valores Faltando

In [48]:
print(base.isna().sum())   # mostre o somatório de linhas com valores vazios para cada coluna

space                                           358714
description                                      18170
neighborhood_overview                           420538
transit                                         425782
house_rules                                     444397
                                                 ...  
maximum_nights_avg_ntm                          301211
number_of_reviews_ltm                           301211
calculated_host_listings_count_entire_homes     301211
calculated_host_listings_count_private_rooms    301211
calculated_host_listings_count_shared_rooms     301211
Length: 65, dtype: int64


In [49]:
# Exclui colunas com mais de 50% (300 mil) valores nulos

for coluna in base:
    if base[coluna].isna().sum() > 300000:
        base = base.drop(coluna, axis = 1)

# 3. Verifica o novo tamanho do DataFrame
print(f"Novo formato do DataFrame: {base.shape}")

Novo formato do DataFrame: (902210, 38)


### Excluir linhas com valores vazios em pelo menos 3 colunas

In [ ]:
# cria uma nova coluna com a contagem de NaNs por linha
base['total_nan_na_linha'] = base.isna().sum(axis=1)

# Pega os índices das linhas que têm mais de 2 NaNs
indices_para_excluir = base[base['total_nan_na_linha'] > 3].index

# Exclui essas linhas pelo índice
base = base.drop(indices_para_excluir)

# Exclui a coluna criada no inicio  para contagem de nanas por linha
base = base.drop(columns = ['total_nan_na_linha'])

print(f"Novo formato do DataFrame: {base.shape}")

Novo formato do DataFrame: (823072, 38)


In [59]:
# Excluir linhas que tenham valor nan

base = base.dropna()
print(f"Novo formato do DataFrame: {base.shape}")

Novo formato do DataFrame: (823072, 38)


### Verificar Tipos de Dados em cada coluna

### Análise Exploratória e Tratar Outliers

### Encoding

### Modelo de Previsão

### Análise do Melhor Modelo

### Ajustes e Melhorias no Melhor Modelo